# LeetCode 582 — Kill Process bằng Python

Link đề: https://leetcode.com/problems/kill-process/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Breadth-First Search, Array, Hash Table

Skeleton LeetCode:

```python
class Solution:
    def killProcess(self, pid: List[int], ppid: List[int], kill: int) -> List[int]:
        ...
```

Ghi chú: đây là bài **Premium** (cần tài khoản trả phí để xem trên LeetCode). Đề trong notebook được viết theo hiểu biết về bài, nên hãy đối chiếu lại nếu bạn có tài khoản Premium. Các ví dụ 3 trở đi trong mục 2 là **tự dựng** (đáp án đã kiểm chứng bằng code), cũng nên đối chiếu lại đề gốc nếu cần. Đề cho phép trả kết quả **theo thứ tự bất kỳ**, nên notebook so sánh sau khi `sort` hoặc bằng hàm kiểm tra tính hợp lệ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có `n` tiến trình tạo thành một **cây** (mỗi tiến trình có đúng một cha, trừ gốc có `ppid = 0`); khi **kill** một tiến trình thì **mọi tiến trình con cháu** của nó cũng bị kill. Hãy liệt kê các `pid` bị kill.
2. Nắm các công cụ Python: hai list **song song** (`pid[i]` và `ppid[i]` là cặp cha–con), `zip`, `dict`, `defaultdict(list)`, `deque` làm hàng đợi, `list` làm ngăn xếp, và giới hạn **độ sâu đệ quy** của Python.
3. Đi từ **quét lại toàn bộ danh sách ở mỗi bước (O(n²))** → **dựng bảng "cha → danh sách con" một lần** → **DFS đệ quy** → **BFS bằng `deque`** → **DFS lặp bằng ngăn xếp** → **đi ngược lên tổ tiên có nhớ (memo)**.
4. Hiểu mấu chốt: đầu vào cho **cha của từng con**, nhưng ta cần **con của từng cha** — đảo chiều quan hệ bằng một bảng băm (hash map) rồi duyệt cây từ `kill` xuống, mỗi tiến trình được thăm đúng một lần.
5. Tránh các lỗi kinh điển: quên chính `kill` trong kết quả, `KeyError` do dùng `dict` thường, nhầm **chỉ số** `i` với **id** `pid[i]`, biến kết quả đặt ở mức class bị dính giữa các lần gọi, và **đệ quy quá sâu** với cây dạng "dây chuyền" (5×10^4 tầng).

## 2. Đề bài & ví dụ

Có `n` tiến trình tạo thành một **cây có gốc**. Bạn được cho hai mảng số nguyên `pid` và `ppid`, trong đó `pid[i]` là ID của tiến trình thứ `i` và `ppid[i]` là ID của **tiến trình cha** của tiến trình thứ `i`.

Mỗi tiến trình có **đúng một** tiến trình cha nhưng có thể có **nhiều** tiến trình con. Chỉ có **một** tiến trình có `ppid[i] = 0`, nghĩa là nó **không có cha** (gốc của cây).

Khi một tiến trình bị **kill**, **tất cả tiến trình con** của nó cũng bị kill (và cứ thế truyền xuống, tức mọi con cháu).

Cho số nguyên `kill` là ID của tiến trình bạn muốn kill, hãy trả về **danh sách ID** của các tiến trình sẽ bị kill. Bạn có thể trả về theo **thứ tự bất kỳ**.

**Ví dụ 1**

- Input: `pid = [1,3,10,5]`, `ppid = [3,0,5,3]`, `kill = 5`
- Output: `[5,10]`
- Giải thích: cây là `3 → {1, 5}`, `5 → {10}`. Kill `5` thì `10` (con của `5`) cũng bị kill. Tiến trình `3` (cha) và `1` (anh em) không bị ảnh hưởng.

**Ví dụ 2**

- Input: `pid = [1]`, `ppid = [0]`, `kill = 1`
- Output: `[1]`

**Ví dụ 3 (tự dựng, đối chiếu lại đề gốc nếu cần)**

- Input: `pid = [1,2,3,4,5,6,7]`, `ppid = [0,1,1,2,2,3,3]`, `kill = 3`
- Output: `[3,6,7]` (thứ tự bất kỳ)

**Ràng buộc:**

- `n == pid.length == ppid.length`
- `1 <= n <= 5 * 10^4`
- `1 <= pid[i] <= 5 * 10^4`
- `0 <= ppid[i] <= 5 * 10^4`
- Chỉ có **một** tiến trình không có cha.
- Mọi giá trị của `pid` là **duy nhất**.
- `kill` được đảm bảo có trong `pid`.

Chú ý: thứ tự trong `pid` là **tùy ý** (cha có thể đứng sau con), nên ta không thể giả định "gặp cha trước con".

In [1]:
from typing import List
from collections import defaultdict, deque
import random
import sys

examples = [
    ([1, 3, 10, 5], [3, 0, 5, 3], 5, [5, 10]),
    ([1], [0], 1, [1]),
    ([1, 2, 3, 4, 5, 6, 7], [0, 1, 1, 2, 2, 3, 3], 3, [3, 6, 7]),
]
for pid, ppid, kill, expected in examples:
    print(f"pid={pid} ppid={ppid} kill={kill} -> kỳ vọng (sau sort) {sorted(expected)}")

pid=[1, 3, 10, 5] ppid=[3, 0, 5, 3] kill=5 -> kỳ vọng (sau sort) [5, 10]
pid=[1] ppid=[0] kill=1 -> kỳ vọng (sau sort) [1]
pid=[1, 2, 3, 4, 5, 6, 7] ppid=[0, 1, 1, 2, 2, 3, 3] kill=3 -> kỳ vọng (sau sort) [3, 6, 7]


## 3. Kiến thức nền

### 3.1. Hai list song song và `zip`

`pid` và `ppid` là hai list **cùng độ dài** mà phần tử cùng vị trí đi thành một cặp: tiến trình `pid[i]` có cha là `ppid[i]`. `zip(pid, ppid)` ghép sẵn từng cặp `(con, cha)`.

In [2]:
pid = [1, 3, 10, 5]
ppid = [3, 0, 5, 3]
for child, parent in zip(pid, ppid):
    print(f"tiến trình {child} có cha là {parent}")

tiến trình 1 có cha là 3
tiến trình 3 có cha là 0
tiến trình 10 có cha là 5
tiến trình 5 có cha là 3


### 3.2. `dict` và `defaultdict(list)`

`dict` là bảng "khóa → giá trị", tra cứu trung bình O(1). Nếu ta cần "cha → **danh sách** con", mỗi khóa ứng với một list. Với `dict` thường, truy cập khóa **chưa có** gây `KeyError`; bạn phải tự tạo list rỗng trước. `defaultdict(list)` làm việc đó giúp: truy cập khóa mới sẽ **tự tạo `[]`**.

In [3]:
g = {}
try:
    g[3].append(1)
except KeyError as e:
    print("dict thường: KeyError", e)

g = defaultdict(list)
g[3].append(1)          # tự tạo g[3] = [] rồi thêm 1
g[3].append(5)
print("defaultdict:", dict(g))
print("khóa chưa có:", g[99], "(lưu ý: tra cứu này cũng TẠO khóa 99)")
print("các khóa:", list(g.keys()))

dict thường: KeyError 3
defaultdict: {3: [1, 5]}
khóa chưa có: [] (lưu ý: tra cứu này cũng TẠO khóa 99)
các khóa: [3, 99]


Hai cách an toàn khi chỉ muốn **đọc** mà không muốn vô tình tạo khóa: `g.get(k, [])` (trả `[]` nếu không có, không tạo khóa) hoặc kiểm tra `if k in g`.

In [4]:
g = {3: [1, 5]}
print(g.get(3, []), g.get(99, []))
print("khóa 99 có bị tạo không?", 99 in g)

[1, 5] []
khóa 99 có bị tạo không? False


### 3.3. Cây, cha, con, gốc

- **Cây có gốc**: có một nút gốc; mỗi nút khác có đúng **một cha**.
- **Con** của `u`: các nút có cha là `u`. **Con cháu (descendants)** của `u`: con, con của con, ... (không kể chính `u`).
- Ở đề này **gốc** là tiến trình có `ppid = 0`. Số `0` không phải là ID của tiến trình nào (vì `pid[i] >= 1`), nó chỉ là dấu "không có cha".

Vì mỗi nút có **một** cha, từ gốc xuống một nút luôn chỉ có **một đường đi** — nên khi duyệt từ `kill` xuống, ta **không bao giờ thăm một nút hai lần**: không cần tập `visited`.

### 3.4. DFS đệ quy (Depth-First Search)

DFS đi **sâu** hết nhánh này rồi mới sang nhánh khác. Viết bằng đệ quy: hàm `dfs(u)` xử lý `u` rồi gọi `dfs(v)` cho từng con `v`.

In [5]:
g = {3: [1, 5], 5: [10]}

def dfs_demo(u, depth=0):
    print("  " * depth + f"thăm {u}")
    for v in g.get(u, []):
        dfs_demo(v, depth + 1)

dfs_demo(3)

thăm 3
  thăm 1
  thăm 5
    thăm 10


### 3.5. BFS (Breadth-First Search) với `deque`

BFS thăm theo **từng tầng**: hết con của gốc, rồi cháu... Dùng **hàng đợi** (vào trước – ra trước). `collections.deque` có `append` (vào cuối) và `popleft()` (ra đầu) đều O(1); dùng `list.pop(0)` thì O(n) — chậm.

In [6]:
q = deque([3])
order = []
while q:
    u = q.popleft()
    order.append(u)
    for v in g.get(u, []):
        q.append(v)
print("thứ tự BFS:", order)

thứ tự BFS: [3, 1, 5, 10]


### 3.6. Ngăn xếp (stack) bằng `list` — DFS không đệ quy

`list.append(x)` đẩy vào đỉnh, `list.pop()` lấy đỉnh ra (cả hai O(1)). Dùng stack thay đệ quy cho DFS: lấy một nút ra, xử lý, rồi đẩy con của nó vào.

In [7]:
stk = [3]
order = []
while stk:
    u = stk.pop()
    order.append(u)
    for v in g.get(u, []):
        stk.append(v)
print("thứ tự DFS lặp:", order)

thứ tự DFS lặp: [3, 5, 10, 1]


### 3.7. Giới hạn đệ quy của Python

Python mặc định chỉ cho đệ quy sâu khoảng **1000** tầng (`sys.getrecursionlimit()`). Vượt quá sẽ có `RecursionError`. Cây "dây chuyền" `1 → 2 → 3 → ... → 5×10^4` sâu **50.000** tầng — DFS đệ quy dễ bị lỗi, nên BFS / DFS lặp an toàn hơn.

In [8]:
print("giới hạn mặc định:", sys.getrecursionlimit())

def depth(n):
    return 0 if n == 0 else 1 + depth(n - 1)

print("depth(100) =", depth(100))
try:
    depth(10**5)
except RecursionError:
    print("depth(10**5): RecursionError (đệ quy quá sâu)")

giới hạn mặc định: 1000
depth(100) = 100
depth(10**5): RecursionError (đệ quy quá sâu)


### 3.8. So sánh kết quả bất kể thứ tự

Vì đề cho phép trả về theo thứ tự bất kỳ, khi test ta so `sorted(a) == sorted(b)` (hoặc dùng `set` nếu biết không trùng).

In [9]:
a = [5, 10]
b = [10, 5]
print(a == b, sorted(a) == sorted(b), set(a) == set(b))

False True True


## 4. Ý tưởng nền tảng / trực giác

Vẽ cây của ví dụ 1: `pid = [1,3,10,5]`, `ppid = [3,0,5,3]`.

| pid[i] | 1 | 3 | 10 | 5 |
|---|---|---|---|---|
| ppid[i] (cha) | 3 | 0 | 5 | 3 |

```
(0)
 └── 3
      ├── 1
      └── 5
           └── 10
```

Kill `5`: bắt đầu với `{5}`. Ai có cha là `5`? Chỉ `10` → thêm `10`. Ai có cha là `10`? Không ai → dừng. Kết quả `{5, 10}`.

Quy tắc: **"đã bị kill" lan từ cha xuống con**. Ta cần biết **con của một tiến trình** — nhưng đề cho ngược lại (cha của từng con). Hãy thử làm đúng như tay người bằng code, rồi đặt "trọng tài" vét cạn: lặp đi lặp lại "nếu cha của một tiến trình đã bị kill thì kill nó" cho tới khi không có gì mới.

In [10]:
def oracle(pid, ppid, kill):
    """Trọng tài: lặp tới khi không thêm được ai (bao đóng - closure)."""
    killed = {kill}
    changed = True
    while changed:
        changed = False
        for child, parent in zip(pid, ppid):
            if parent in killed and child not in killed:
                killed.add(child)
                changed = True
    return sorted(killed)

for pid, ppid, kill, expected in examples:
    assert oracle(pid, ppid, kill) == sorted(expected)
    print(pid, ppid, kill, "->", oracle(pid, ppid, kill))

[1, 3, 10, 5] [3, 0, 5, 3] 5 -> [5, 10]
[1] [0] 1 -> [1]
[1, 2, 3, 4, 5, 6, 7] [0, 1, 1, 2, 2, 3, 3] 3 -> [3, 6, 7]


`oracle` rất chậm (nhiều lượt quét), nhưng định nghĩa của nó **gần như là đề bài viết thành code**, nên ta tin nó để kiểm chứng các lời giải khác. Chú ý: ý tưởng này khác ý tưởng "duyệt cây từ `kill` xuống" của các cách sau, nên oracle là nguồn đối chiếu độc lập.

## 5. Cách 1 — Brute Force: mỗi lần quét lại toàn bộ danh sách

**Ý tưởng.** Giữ một danh sách `killed` (bắt đầu `[kill]`) và một con trỏ `k` tới tiến trình đang xử lý. Với mỗi tiến trình `cur` lấy ra, **quét toàn bộ** `pid/ppid` để tìm các tiến trình có `ppid[i] == cur`; mỗi cái tìm được thì thêm vào `killed` (nó sẽ được xử lý ở các bước sau). Dừng khi `k` đuổi kịp cuối `killed`.

Cách này đúng (mỗi tiến trình bị kill được "mở rộng" đúng một lần) nhưng với **mỗi** tiến trình bị kill phải quét `n` phần tử.

In [11]:
def brute_scan(pid, ppid, kill):
    n = len(pid)
    killed = [kill]
    k = 0
    while k < len(killed):
        cur = killed[k]
        k += 1
        for i in range(n):              # quét TOÀN BỘ để tìm con của cur
            if ppid[i] == cur:
                killed.append(pid[i])
    return killed

for pid, ppid, kill, expected in examples:
    got = brute_scan(pid, ppid, kill)
    assert sorted(got) == sorted(expected)
    print(pid, ppid, kill, "->", got)

[1, 3, 10, 5] [3, 0, 5, 3] 5 -> [5, 10]
[1] [0] 1 -> [1]
[1, 2, 3, 4, 5, 6, 7] [0, 1, 1, 2, 2, 3, 3] 3 -> [3, 6, 7]


**Độ phức tạp.** Nếu `m` tiến trình bị kill, mỗi cái quét `n` phần tử: **Time O(n·m)**, tệ nhất `m ≈ n` (kill gốc) → **O(n²)** = `2.5 × 10^9` phép so sánh với `n = 5×10^4` — chắc chắn quá thời gian. **Space O(m)** cho `killed`.

**Nút thắt.** Việc "tìm con của `cur`" phải đi qua cả `ppid` mỗi lần. Nếu **chuẩn bị trước** một bảng "cha → danh sách con" (chỉ tốn một lần quét), thì lấy con của bất kỳ tiến trình nào chỉ còn O(1) — đó là bước đột phá.

## 6. Ý tưởng thuật toán / bước đột phá

Đảo chiều quan hệ: từ "(con, cha)" sang "cha → [các con]". Chỉ cần **một** vòng quét:

```
g = defaultdict(list)
for con, cha in zip(pid, ppid):
    g[cha].append(con)
```

Sau đó `g[u]` là danh sách con của `u`. Việc còn lại là **duyệt đồ thị từ `kill`**: DFS hoặc BFS — mỗi nút thăm một lần, mỗi cạnh dùng một lần → **O(n)**.

Hãy xem `g` được dựng ra sao với ví dụ 3.

In [12]:
pid = [1, 2, 3, 4, 5, 6, 7]
ppid = [0, 1, 1, 2, 2, 3, 3]
g = defaultdict(list)
for child, parent in zip(pid, ppid):
    g[parent].append(child)
print("bảng cha -> con:")
for parent in sorted(g):
    print(f"  {parent} -> {g[parent]}")
print("con của 3:", g[3])

bảng cha -> con:
  0 -> [1]
  1 -> [2, 3]
  2 -> [4, 5]
  3 -> [6, 7]
con của 3: [6, 7]


Lưu ý `g[0] = [1]`: khóa `0` là "cha giả" của gốc — vô hại, vì `kill` luôn là một `pid` thật (`>= 1`) nên ta không bao giờ duyệt qua `0`.

## 7. Các cách tiếp theo

Tổng cộng 5 cách (cách 1 đã ở mục 5):

| # | Cách | Time | Space |
|---|---|---|---|
| 1 | Quét lại toàn bộ mỗi bước (brute force) | O(n·m) ≤ O(n²) | O(m) |
| 2 | Bảng con + DFS đệ quy | O(n) | O(n) (+ độ sâu đệ quy) |
| 3 | Bảng con + BFS với `deque` | O(n) | O(n) |
| 4 | Bảng con + DFS lặp bằng ngăn xếp | O(n) | O(n) |
| 5 | Đi ngược lên tổ tiên có nhớ (memo) | O(n) | O(n) |

### 7.1. Cách 2 — Bảng con + DFS đệ quy

**Giải thích từng đoạn.**

1. `g = defaultdict(list)`; mỗi cặp `(con, cha)` thì `g[cha].append(con)`.
2. Hàm `dfs(u)`: thêm `u` vào `ans`, rồi với mỗi `v` trong `g[u]` gọi `dfs(v)`.
3. Gọi `dfs(kill)` và trả `ans`.

`ans` là list **dùng chung** ở hàm ngoài; hàm trong `dfs` chỉ `append` (không gán lại `ans`) nên không cần `nonlocal`. Điểm yếu: độ sâu đệ quy bằng độ sâu của cây.

In [13]:
def dfs_recursive(pid, ppid, kill):
    g = defaultdict(list)
    for child, parent in zip(pid, ppid):
        g[parent].append(child)
    ans = []

    def dfs(u):
        ans.append(u)
        for v in g[u]:
            dfs(v)

    dfs(kill)
    return ans

for pid, ppid, kill, expected in examples:
    got = dfs_recursive(pid, ppid, kill)
    assert sorted(got) == sorted(expected)
    print(pid, ppid, kill, "->", got)

[1, 3, 10, 5] [3, 0, 5, 3] 5 -> [5, 10]
[1] [0] 1 -> [1]
[1, 2, 3, 4, 5, 6, 7] [0, 1, 1, 2, 2, 3, 3] 3 -> [3, 6, 7]


In [14]:
# Code nộp LeetCode — Cách 2: DFS đệ quy
class Solution:
    def killProcess(self, pid: List[int], ppid: List[int], kill: int) -> List[int]:
        g = defaultdict(list)
        for child, parent in zip(pid, ppid):
            g[parent].append(child)
        ans = []

        def dfs(u: int) -> None:
            ans.append(u)
            for v in g[u]:
                dfs(v)

        dfs(kill)
        return ans

for pid, ppid, kill, expected in examples:
    assert sorted(Solution().killProcess(pid, ppid, kill)) == sorted(expected)
print("Cách 2 qua ví dụ đề")

Cách 2 qua ví dụ đề


**Độ phức tạp.** Dựng bảng O(n); DFS thăm mỗi tiến trình bị kill một lần và duyệt mỗi cạnh một lần → **Time O(n)**. **Space O(n)** cho bảng, `ans` và ngăn xếp đệ quy (có thể sâu tới `n`). Trên cây dây chuyền sâu 5×10^4, Python mặc định có thể `RecursionError` — ta sẽ chứng kiến ở mục 12–13.

### 7.2. Cách 3 — Bảng con + BFS với `deque`

**Giải thích từng đoạn.**

1. Dựng `g` như trước.
2. `q = deque([kill])`, `ans = []`.
3. Lặp `while q`: lấy `u = q.popleft()`, ghi vào `ans`, đẩy mọi con `g[u]` vào `q`.

BFS không dùng đệ quy nên không sợ cây sâu, và thứ tự kết quả là theo tầng (kill, các con, các cháu...). Mỗi nút vào hàng đợi đúng một lần (cây không có chu trình, mỗi nút một cha).

`q.extend(g[u])` thêm **tất cả** con của `u` vào cuối hàng đợi chỉ bằng một lệnh; còn `ans.append(u)` làm khi lấy `u` ra khỏi hàng đợi.

In [15]:
def bfs_queue(pid, ppid, kill):
    g = defaultdict(list)
    for child, parent in zip(pid, ppid):
        g[parent].append(child)
    ans = []
    q = deque([kill])
    while q:
        u = q.popleft()
        ans.append(u)
        q.extend(g[u])
    return ans

for pid, ppid, kill, expected in examples:
    got = bfs_queue(pid, ppid, kill)
    assert sorted(got) == sorted(expected)
    print(pid, ppid, kill, "->", got)

[1, 3, 10, 5] [3, 0, 5, 3] 5 -> [5, 10]
[1] [0] 1 -> [1]
[1, 2, 3, 4, 5, 6, 7] [0, 1, 1, 2, 2, 3, 3] 3 -> [3, 6, 7]


In [16]:
# Code nộp LeetCode — Cách 3: BFS với deque
class Solution:
    def killProcess(self, pid: List[int], ppid: List[int], kill: int) -> List[int]:
        g = defaultdict(list)
        for child, parent in zip(pid, ppid):
            g[parent].append(child)
        ans = []
        q = deque([kill])
        while q:
            u = q.popleft()
            ans.append(u)
            q.extend(g[u])
        return ans

for pid, ppid, kill, expected in examples:
    assert sorted(Solution().killProcess(pid, ppid, kill)) == sorted(expected)
print("Cách 3 qua ví dụ đề")

Cách 3 qua ví dụ đề


**Độ phức tạp.** **Time O(n)**: dựng bảng O(n) + mỗi nút bị kill vào/ra hàng đợi một lần, mỗi cạnh xét một lần. **Space O(n)**: bảng `g`, `ans`, hàng đợi (tối đa bề ngang cây).

### 7.3. Cách 4 — Bảng con + DFS lặp bằng ngăn xếp

Giống DFS đệ quy nhưng **ta tự quản lý** ngăn xếp: lấy `u = stk.pop()`, ghi `u`, đẩy các con vào. Không có giới hạn đệ quy; thứ tự thăm hơi khác (con cuối được xử lý trước) — không sao vì đề không đòi thứ tự. Nếu muốn khớp đúng thứ tự DFS đệ quy thì đẩy các con theo thứ tự **đảo ngược** (`reversed(g[u])`).

In [17]:
def dfs_stack(pid, ppid, kill):
    g = defaultdict(list)
    for child, parent in zip(pid, ppid):
        g[parent].append(child)
    ans = []
    stk = [kill]
    while stk:
        u = stk.pop()
        ans.append(u)
        for v in reversed(g[u]):
            stk.append(v)
    return ans

for pid, ppid, kill, expected in examples:
    got = dfs_stack(pid, ppid, kill)
    assert sorted(got) == sorted(expected)
    print(pid, ppid, kill, "->", got, "| DFS đệ quy:", dfs_recursive(pid, ppid, kill))
    assert got == dfs_recursive(pid, ppid, kill)    # nhờ reversed, cùng thứ tự DFS

[1, 3, 10, 5] [3, 0, 5, 3] 5 -> [5, 10] | DFS đệ quy: [5, 10]
[1] [0] 1 -> [1] | DFS đệ quy: [1]
[1, 2, 3, 4, 5, 6, 7] [0, 1, 1, 2, 2, 3, 3] 3 -> [3, 6, 7] | DFS đệ quy: [3, 6, 7]


In [18]:
# Code nộp LeetCode — Cách 4: DFS lặp bằng ngăn xếp
class Solution:
    def killProcess(self, pid: List[int], ppid: List[int], kill: int) -> List[int]:
        g = defaultdict(list)
        for child, parent in zip(pid, ppid):
            g[parent].append(child)
        ans = []
        stk = [kill]
        while stk:
            u = stk.pop()
            ans.append(u)
            for v in reversed(g[u]):
                stk.append(v)
        return ans

for pid, ppid, kill, expected in examples:
    assert sorted(Solution().killProcess(pid, ppid, kill)) == sorted(expected)
print("Cách 4 qua ví dụ đề")

Cách 4 qua ví dụ đề


**Độ phức tạp.** **Time O(n)**, **Space O(n)** (bảng `g`, `ans`, ngăn xếp tối đa bằng số nút đang chờ xử lý). Ưu điểm so với cách 2: không giới hạn độ sâu.

### 7.4. Cách 5 — Đi ngược lên tổ tiên, có nhớ (memo)

**Ý tưởng hoàn toàn khác**: không dựng "cha → con", mà dựng "con → cha" (chính là dữ liệu đề cho) rồi hỏi từng tiến trình: "**`kill` có phải là tổ tiên của bạn không?**" (hoặc chính bạn là `kill`). Bạn bị kill ⇔ đi ngược theo chuỗi cha sẽ gặp `kill` trước khi chạm `0`.

Hỏi riêng từng tiến trình có thể tốn O(độ sâu), tổng O(n²) trên dây chuyền. Để tránh, **nhớ** kết quả `memo[x]` (True/False) cho mọi nút đã đi qua: khi đi ngược gặp một nút đã biết, gán luôn kết quả cho cả đường vừa đi.

**Giải thích từng đoạn.**

1. `parent = dict(zip(pid, ppid))`.
2. `memo = {kill: True, 0: False}`: `kill` chắc chắn bị kill; "cha giả" `0` nghĩa là đã tới đỉnh mà không gặp `kill`.
3. Với mỗi `p` theo thứ tự đầu vào: đi ngược `cur = p`, ghi các nút vào `path` cho tới khi `cur` đã nằm trong `memo`; kết quả `res = memo[cur]`; gán `memo[x] = res` cho cả `path`.
4. Nếu `memo[p]` là True thì thêm `p` vào kết quả.

Mỗi nút được đưa vào `path` đúng một lần (sau đó nó đã có trong `memo`) nên tổng là O(n). Dùng vòng `while` (không đệ quy) nên không sợ cây sâu. Thứ tự kết quả là theo thứ tự `pid` đầu vào — đề cho phép.

In [19]:
def ancestor_memo(pid, ppid, kill):
    parent = dict(zip(pid, ppid))
    memo = {kill: True, 0: False}
    ans = []
    for p in pid:
        path = []
        cur = p
        while cur not in memo:          # đi ngược lên cho tới nút đã biết kết quả
            path.append(cur)
            cur = parent[cur]
        res = memo[cur]
        for x in path:                  # gán luôn cho cả đoạn đường vừa đi
            memo[x] = res
        if memo[p]:
            ans.append(p)
    return ans

for pid, ppid, kill, expected in examples:
    got = ancestor_memo(pid, ppid, kill)
    assert sorted(got) == sorted(expected)
    print(pid, ppid, kill, "->", got)

[1, 3, 10, 5] [3, 0, 5, 3] 5 -> [10, 5]
[1] [0] 1 -> [1]
[1, 2, 3, 4, 5, 6, 7] [0, 1, 1, 2, 2, 3, 3] 3 -> [3, 6, 7]


In [20]:
# Code nộp LeetCode — Cách 5: tổ tiên có nhớ
class Solution:
    def killProcess(self, pid: List[int], ppid: List[int], kill: int) -> List[int]:
        parent = dict(zip(pid, ppid))
        memo = {kill: True, 0: False}
        ans = []
        for p in pid:
            path = []
            cur = p
            while cur not in memo:
                path.append(cur)
                cur = parent[cur]
            res = memo[cur]
            for x in path:
                memo[x] = res
            if memo[p]:
                ans.append(p)
        return ans

for pid, ppid, kill, expected in examples:
    assert sorted(Solution().killProcess(pid, ppid, kill)) == sorted(expected)
print("Cách 5 qua ví dụ đề")

Cách 5 qua ví dụ đề


**Độ phức tạp.** **Time O(n)** (mỗi nút vào `path` một lần). **Space O(n)** cho `parent`, `memo`, `path`. Điểm đáng chú ý: cách này **luôn quét cả n tiến trình** kể cả khi chỉ kill một lá, trong khi BFS/DFS chỉ đi qua những nút bị kill (sau khi dựng bảng).

## 8. TRACE từng bước

Trace **BFS** (cách 3) với `pid = [1,2,3,4,5,6,7]`, `ppid = [0,1,1,2,2,3,3]`, `kill = 2`. Cây:

```
1
├── 2
│   ├── 4
│   └── 5
└── 3
    ├── 6
    └── 7
```

Bảng `g`: `0→[1]`, `1→[2,3]`, `2→[4,5]`, `3→[6,7]`.

| bước | hàng đợi `q` trước | `u = popleft()` | `g[u]` | `q` sau | `ans` |
|---|---|---|---|---|---|
| 0 | [2] | | | | [] |
| 1 | [2] | 2 | [4,5] | [4,5] | [2] |
| 2 | [4,5] | 4 | [] | [5] | [2,4] |
| 3 | [5] | 5 | [] | [] | [2,4,5] |
| dừng | [] | | | | [2,4,5] |

Chú ý: `3`, `6`, `7` (nhánh anh em) và `1` (cha) **không** bao giờ vào hàng đợi.

In [21]:
def trace_bfs(pid, ppid, kill):
    g = defaultdict(list)
    for child, parent in zip(pid, ppid):
        g[parent].append(child)
    print("g =", {k: v for k, v in sorted(g.items())})
    ans = []
    q = deque([kill])
    step = 0
    while q:
        step += 1
        before = list(q)
        u = q.popleft()
        ans.append(u)
        kids = g[u]
        q.extend(kids)
        print(f"bước {step}: q trước={before} | lấy u={u} | g[u]={kids} | q sau={list(q)} | ans={ans}")
    print("kết quả:", ans)
    return ans

res = trace_bfs([1, 2, 3, 4, 5, 6, 7], [0, 1, 1, 2, 2, 3, 3], 2)
assert sorted(res) == [2, 4, 5]

g = {0: [1], 1: [2, 3], 2: [4, 5], 3: [6, 7]}
bước 1: q trước=[2] | lấy u=2 | g[u]=[4, 5] | q sau=[4, 5] | ans=[2]
bước 2: q trước=[4, 5] | lấy u=4 | g[u]=[] | q sau=[5] | ans=[2, 4]
bước 3: q trước=[5] | lấy u=5 | g[u]=[] | q sau=[] | ans=[2, 4, 5]
kết quả: [2, 4, 5]


Trace **DFS lặp bằng ngăn xếp** (cách 4) với `kill = 1` trên cùng cây (kill gốc → mọi người bị kill). Vì ta đẩy con theo `reversed`, con đầu tiên nằm trên đỉnh và được xử lý trước:

| bước | `stk` trước | `u = pop()` | các con (g[u]) | `stk` sau | `ans` |
|---|---|---|---|---|---|
| 1 | [1] | 1 | [2,3] | [3,2] | [1] |
| 2 | [3,2] | 2 | [4,5] | [3,5,4] | [1,2] |
| 3 | [3,5,4] | 4 | [] | [3,5] | [1,2,4] |
| 4 | [3,5] | 5 | [] | [3] | [1,2,4,5] |
| 5 | [3] | 3 | [6,7] | [7,6] | [1,2,4,5,3] |
| 6 | [7,6] | 6 | [] | [7] | [1,2,4,5,3,6] |
| 7 | [7] | 7 | [] | [] | [1,2,4,5,3,6,7] |

In [22]:
def trace_stack(pid, ppid, kill):
    g = defaultdict(list)
    for child, parent in zip(pid, ppid):
        g[parent].append(child)
    ans = []
    stk = [kill]
    step = 0
    while stk:
        step += 1
        before = list(stk)
        u = stk.pop()
        ans.append(u)
        kids = g[u]
        for v in reversed(kids):
            stk.append(v)
        print(f"bước {step}: stk trước={before} | pop u={u} | con={kids} | stk sau={stk} | ans={ans}")
    return ans

res = trace_stack([1, 2, 3, 4, 5, 6, 7], [0, 1, 1, 2, 2, 3, 3], 1)
assert res == [1, 2, 4, 5, 3, 6, 7]

bước 1: stk trước=[1] | pop u=1 | con=[2, 3] | stk sau=[3, 2] | ans=[1]
bước 2: stk trước=[3, 2] | pop u=2 | con=[4, 5] | stk sau=[3, 5, 4] | ans=[1, 2]
bước 3: stk trước=[3, 5, 4] | pop u=4 | con=[] | stk sau=[3, 5] | ans=[1, 2, 4]
bước 4: stk trước=[3, 5] | pop u=5 | con=[] | stk sau=[3] | ans=[1, 2, 4, 5]
bước 5: stk trước=[3] | pop u=3 | con=[6, 7] | stk sau=[7, 6] | ans=[1, 2, 4, 5, 3]
bước 6: stk trước=[7, 6] | pop u=6 | con=[] | stk sau=[7] | ans=[1, 2, 4, 5, 3, 6]
bước 7: stk trước=[7] | pop u=7 | con=[] | stk sau=[] | ans=[1, 2, 4, 5, 3, 6, 7]


Cuối cùng trace **cách 5 (tổ tiên có nhớ)** với `pid = [4, 6, 2, 1, 3]`, `ppid = [2, 3, 1, 0, 1]`, `kill = 2` — chú ý thứ tự đầu vào **không** là cha trước con (`4` có cha `2` nhưng `2` đứng sau). Cây: `1 → {2, 3}`, `2 → {4}`, `3 → {6}`. Kỳ vọng `{2, 4}`.

In [23]:
def trace_ancestor(pid, ppid, kill):
    parent = dict(zip(pid, ppid))
    memo = {kill: True, 0: False}
    ans = []
    print("parent =", parent, "| memo ban đầu =", memo)
    for p in pid:
        path, cur = [], p
        while cur not in memo:
            path.append(cur)
            cur = parent[cur]
        res = memo[cur]
        for x in path:
            memo[x] = res
        if memo[p]:
            ans.append(p)
        print(f"p={p}: đi ngược qua {path} tới {cur} (memo={res}) -> gán cả đường = {res} | bị kill? {memo[p]} | memo={memo}")
    print("kết quả:", ans)
    return ans

res = trace_ancestor([4, 6, 2, 1, 3], [2, 3, 1, 0, 1], 2)
assert sorted(res) == [2, 4]

parent = {4: 2, 6: 3, 2: 1, 1: 0, 3: 1} | memo ban đầu = {2: True, 0: False}
p=4: đi ngược qua [4] tới 2 (memo=True) -> gán cả đường = True | bị kill? True | memo={2: True, 0: False, 4: True}
p=6: đi ngược qua [6, 3, 1] tới 0 (memo=False) -> gán cả đường = False | bị kill? False | memo={2: True, 0: False, 4: True, 6: False, 3: False, 1: False}
p=2: đi ngược qua [] tới 2 (memo=True) -> gán cả đường = True | bị kill? True | memo={2: True, 0: False, 4: True, 6: False, 3: False, 1: False}
p=1: đi ngược qua [] tới 1 (memo=False) -> gán cả đường = False | bị kill? False | memo={2: True, 0: False, 4: True, 6: False, 3: False, 1: False}
p=3: đi ngược qua [] tới 3 (memo=False) -> gán cả đường = False | bị kill? False | memo={2: True, 0: False, 4: True, 6: False, 3: False, 1: False}
kết quả: [4, 2]


## 9. Code nộp LeetCode

Ta nộp **cách 3 (BFS với `deque`)**: O(n), không sợ cây sâu, code ngắn và dễ đọc. Trên LeetCode, `List`, `defaultdict`, `deque` đã được import sẵn; trong notebook ta import ở đầu.

In [24]:
class Solution:
    def killProcess(self, pid: List[int], ppid: List[int], kill: int) -> List[int]:
        g = defaultdict(list)
        for child, parent in zip(pid, ppid):
            g[parent].append(child)
        ans = []
        q = deque([kill])
        while q:
            u = q.popleft()
            ans.append(u)
            q.extend(g[u])
        return ans


sol = Solution()
for pid, ppid, kill, expected in examples:
    got = sol.killProcess(pid, ppid, kill)
    print(pid, ppid, kill, "->", got, "(kỳ vọng", sorted(expected), "sau sort)")
    assert sorted(got) == sorted(expected)

[1, 3, 10, 5] [3, 0, 5, 3] 5 -> [5, 10] (kỳ vọng [5, 10] sau sort)
[1] [0] 1 -> [1] (kỳ vọng [1] sau sort)
[1, 2, 3, 4, 5, 6, 7] [0, 1, 1, 2, 2, 3, 3] 3 -> [3, 6, 7] (kỳ vọng [3, 6, 7] sau sort)


## 10. Vì sao đúng

**Quan hệ.** Gọi `D(kill)` là tập mọi tiến trình bị kill = `kill` cùng mọi con cháu của nó. Theo định nghĩa đệ quy: `D(u) = {u} ∪ ⋃ D(v)` với `v` chạy qua các con của `u`.

**Bảng `g` đúng.** `g[u]` chứa đúng các `pid[i]` có `ppid[i] = u`, tức đúng các con của `u` — chứng minh trực tiếp từ vòng dựng bảng (mỗi cặp được thêm một lần vào danh sách của cha nó).

**BFS/DFS trả đúng `D(kill)`** (quy nạp theo độ sâu của nút so với `kill`):

- *Đủ:* nút ở độ sâu 0 (`kill`) được thêm đầu tiên. Nếu mọi nút ở độ sâu `d` đã được lấy ra, thì khi lấy ra nút `u` ta đẩy toàn bộ con của `u` (độ sâu `d+1`) vào hàng đợi/ngăn xếp → mọi nút độ sâu `d+1` cũng sẽ được lấy ra. Vậy mọi con cháu đều có mặt.
- *Không thừa:* mọi nút vào hàng đợi đều là con của một nút đã có mặt, nên nó nằm trong `D(kill)`.
- *Không trùng:* trong **cây**, mỗi nút có đúng một cha, nên nút `v` chỉ xuất hiện trong danh sách `g[cha_của_v]` — chỉ được đẩy vào đúng một lần, khi `cha_của_v` được xử lý (và `cha_của_v` cũng chỉ được xử lý một lần theo lập luận quy nạp). Do đó không cần `visited`.

**Cách 5 đúng:** tiến trình `p` thuộc `D(kill)` ⇔ chuỗi cha `p, parent[p], parent[parent[p]], ...` chứa `kill`. Chuỗi này kết thúc tại `0` (gốc có cha `0`) và là đường đi duy nhất lên gốc; `memo[x]` lưu "chuỗi cha của `x` có chứa `kill` không" — dùng được lại cho mọi nút khác đi qua `x`, vì chuỗi cha của một nút là phần đuôi của chuỗi cha của con nó.

**Cách 1 đúng:** mỗi nút `cur` bị kill được "mở rộng" đúng một lần bằng cách quét tìm mọi con; vì mỗi nút có một cha nên không thêm trùng.

## 11. Time / Space Complexity

Gọi `n` là số tiến trình, `m` là số tiến trình bị kill (`m ≤ n`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Quét lại toàn bộ | O(n·m) ≤ O(n²) | O(m) | kill gốc với n = 5×10^4: khoảng 2,5 tỷ phép so sánh |
| 2. Bảng + DFS đệ quy | O(n) | O(n) | độ sâu đệ quy tới n: nguy cơ RecursionError |
| 3. Bảng + BFS `deque` | O(n) | O(n) | **cách nộp**, an toàn với cây sâu |
| 4. Bảng + DFS ngăn xếp | O(n) | O(n) | tương đương BFS, thứ tự giống DFS đệ quy |
| 5. Tổ tiên có nhớ | O(n) | O(n) | luôn quét cả n tiến trình |

Dựng bảng `g` là O(n); phần duyệt chỉ tốn O(m). Vì không ai nhanh hơn O(n) được (phải đọc đầu vào), O(n) là tối ưu.

## 12. Bộ test

Ta cần một **hàm kiểm tra tính hợp lệ** vì đề cho phép thứ tự bất kỳ: kết quả hợp lệ khi nó **không trùng**, và tập các phần tử bằng đúng tập do `oracle` (bao đóng) tính ra.

In [25]:
def is_valid(result, pid, ppid, kill):
    return len(result) == len(set(result)) and sorted(result) == oracle(pid, ppid, kill)

all_ways = {
    "brute_scan": brute_scan,
    "dfs_recursive": dfs_recursive,
    "bfs_queue": bfs_queue,
    "dfs_stack": dfs_stack,
    "ancestor_memo": ancestor_memo,
}

tests = [
    # ví dụ đề
    ([1, 3, 10, 5], [3, 0, 5, 3], 5, [5, 10]),
    ([1], [0], 1, [1]),
    # ví dụ tự dựng
    ([1, 2, 3, 4, 5, 6, 7], [0, 1, 1, 2, 2, 3, 3], 3, [3, 6, 7]),
    # kill gốc: tất cả
    ([1, 2, 3, 4, 5, 6, 7], [0, 1, 1, 2, 2, 3, 3], 1, [1, 2, 3, 4, 5, 6, 7]),
    # kill một lá: chỉ chính nó
    ([1, 2, 3, 4, 5, 6, 7], [0, 1, 1, 2, 2, 3, 3], 6, [6]),
    # thứ tự đầu vào: con đứng trước cha
    ([6, 4, 3, 2, 1], [3, 2, 1, 1, 0], 2, [2, 4]),
    # dây chuyền 1 -> 2 -> 3 -> 4, kill giữa
    ([4, 3, 2, 1], [3, 2, 1, 0], 2, [2, 3, 4]),
    # hình sao: gốc 10 có nhiều con
    ([10, 1, 2, 3, 4], [0, 10, 10, 10, 10], 10, [10, 1, 2, 3, 4]),
    # ID lớn, không liên tục
    ([50000, 25000, 49999], [0, 50000, 25000], 25000, [25000, 49999]),
]
for pid, ppid, kill, expected in tests:
    assert oracle(pid, ppid, kill) == sorted(expected), (pid, ppid, kill)
    for name, fn in all_ways.items():
        got = fn(list(pid), list(ppid), kill)
        assert is_valid(got, pid, ppid, kill), (name, pid, ppid, kill, got)
        assert sorted(got) == sorted(expected), (name, got)
print("OK:", len(tests), "ca cố định, 5 cách khớp oracle")

OK: 9 ca cố định, 5 cách khớp oracle


### Test ngẫu nhiên nhỏ (so với oracle)

Dựng cây ngẫu nhiên **đúng ràng buộc**: `pid` là các số khác nhau trong `[1, 60]`, một gốc `ppid = 0`, các nút còn lại chọn cha ngẫu nhiên trong những nút đã xuất hiện; sau đó **xáo thứ tự** các cặp (con có thể đứng trước cha). `kill` là một `pid` bất kỳ. Seed cố định.

In [26]:
def random_tree(n, max_id, rng):
    ids = rng.sample(range(1, max_id + 1), n)
    # ids[0] là gốc; ids[k] chọn cha trong ids[:k] -> chắc chắn là cây
    pairs = [(ids[0], 0)]
    for k in range(1, n):
        pairs.append((ids[k], rng.choice(ids[:k])))
    rng.shuffle(pairs)
    pid = [c for c, _ in pairs]
    ppid = [p for _, p in pairs]
    return pid, ppid

rng = random.Random(582)
cnt = 0
for _ in range(2000):
    n = rng.randint(1, 25)
    pid, ppid = random_tree(n, 60, rng)
    kill = rng.choice(pid)
    for name, fn in all_ways.items():
        got = fn(list(pid), list(ppid), kill)
        assert is_valid(got, pid, ppid, kill), (name, pid, ppid, kill, got)
    cnt += 1
print("OK:", cnt, "cây ngẫu nhiên nhỏ")

OK: 2000 cây ngẫu nhiên nhỏ


### Test cây lớn (so các cách O(n) với nhau)

Cây ngẫu nhiên `n = 5×10^4` (đúng cận đề, `pid` trong `[1, 5×10^4]` nên `pid` là một hoán vị của `1..50000`). Cách 1 quá chậm nên bỏ qua; cách 2 (đệ quy) dùng được vì cây ngẫu nhiên khá nông (độ sâu cỡ `log n`). Đối chiếu bằng `sorted(...)` và kiểm tra không trùng.

In [27]:
rng = random.Random(5820)
N = 50000
pid, ppid = random_tree(N, 50000, rng)
for kill in (pid[0], pid[1], pid[-1], rng.choice(pid)):
    outs = {}
    for name in ("dfs_recursive", "bfs_queue", "dfs_stack", "ancestor_memo"):
        got = all_ways[name](pid, ppid, kill)
        assert len(got) == len(set(got)), name
        outs[name] = sorted(got)
    ref = outs["bfs_queue"]
    assert all(v == ref for v in outs.values())
    print(f"kill={kill}: số tiến trình bị kill = {len(ref)}")

kill=9408: số tiến trình bị kill = 2
kill=9224: số tiến trình bị kill = 2
kill=31672: số tiến trình bị kill = 3
kill=30848: số tiến trình bị kill = 1


### Hai cây cực đoan: dây chuyền sâu và hình sao rộng

- **Dây chuyền** `1 → 2 → ... → 50000`: độ sâu 50.000 → thử thách cho đệ quy. Kill ở giữa.
- **Hình sao**: một gốc có 49.999 con → danh sách con rất dài.

In [28]:
N = 50000
chain_pid = list(range(1, N + 1))
chain_ppid = [0] + list(range(1, N))          # cha của i+1 là i
kill = 20001
want = list(range(20001, N + 1))
for name in ("bfs_queue", "dfs_stack", "ancestor_memo"):
    got = all_ways[name](chain_pid, chain_ppid, kill)
    assert sorted(got) == want, name
print("dây chuyền: bfs, stack, memo OK, số bị kill =", len(want))

try:
    all_ways["dfs_recursive"](chain_pid, chain_ppid, kill)
    print("dây chuyền: DFS đệ quy chạy được")
except RecursionError:
    print("dây chuyền: DFS đệ quy bị RecursionError (mặc định ~1000 tầng)")

star_pid = list(range(1, N + 1))
star_ppid = [0] + [1] * (N - 1)               # gốc 1, mọi nút khác là con của 1
for name in ("dfs_recursive", "bfs_queue", "dfs_stack", "ancestor_memo"):
    got = all_ways[name](star_pid, star_ppid, 1)
    assert sorted(got) == star_pid, name
    got = all_ways[name](star_pid, star_ppid, 777)
    assert got == [777], name
print("hình sao: 4 cách OK (kill gốc: 50000 tiến trình, kill lá: 1 tiến trình)")

dây chuyền: bfs, stack, memo OK, số bị kill = 30000
dây chuyền: DFS đệ quy bị RecursionError (mặc định ~1000 tầng)
hình sao: 4 cách OK (kill gốc: 50000 tiến trình, kill lá: 1 tiến trình)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quét lại toàn bộ danh sách mỗi lần (chậm)

Mỗi tiến trình bị kill lại đi qua cả `ppid`: **O(n²)** khi kill gốc. Ta đếm số phép so sánh thay vì đo giờ để thấy rõ chênh lệch trên dây chuyền `n = 2000`.

In [29]:
def count_ops_scan(pid, ppid, kill):
    ops, killed, k = 0, [kill], 0
    while k < len(killed):
        cur = killed[k]; k += 1
        for i in range(len(pid)):
            ops += 1
            if ppid[i] == cur:
                killed.append(pid[i])
    return ops

def count_ops_table(pid, ppid, kill):
    ops = len(pid)                       # một vòng dựng bảng
    g = defaultdict(list)
    for c, p in zip(pid, ppid):
        g[p].append(c)
    stk = [kill]
    while stk:
        u = stk.pop()
        for v in g[u]:
            ops += 1                     # mỗi cạnh một lần
            stk.append(v)
    return ops

n = 2000
cp, cpp = list(range(1, n + 1)), [0] + list(range(1, n))
a, b = count_ops_scan(cp, cpp, 1), count_ops_table(cp, cpp, 1)
print("quét lại:", a, "phép so sánh | dùng bảng:", b)
assert a == n * n and b < 2 * n

quét lại: 4000000 phép so sánh | dùng bảng: 3999


**Vì sao sai (chậm)?** Việc "tìm con của `u`" đáng lẽ chỉ tốn O(số con) nhưng ở đây tốn O(n). **Cách đúng:** dựng `g` một lần (mục 6).

### Lỗi 2 — Quên chính tiến trình `kill` trong kết quả

Có bạn chỉ thêm **các con** của `kill` vào kết quả, vì nghĩ "kill ai thì liệt kê những ai bị ảnh hưởng". Đề yêu cầu **gồm cả chính `kill`**.

In [30]:
def wrong_missing_self(pid, ppid, kill):
    g = defaultdict(list)
    for c, p in zip(pid, ppid):
        g[p].append(c)
    ans = []
    q = deque(g[kill])                   # SAI: bắt đầu từ các con, bỏ quên kill
    while q:
        u = q.popleft()
        ans.append(u)
        q.extend(g[u])
    return ans

pid, ppid, kill = [1, 3, 10, 5], [3, 0, 5, 3], 5
print("sai    :", wrong_missing_self(pid, ppid, kill), "| đúng:", bfs_queue(pid, ppid, kill))
assert 5 not in wrong_missing_self(pid, ppid, kill)
assert 5 in bfs_queue(pid, ppid, kill)

sai    : [10] | đúng: [5, 10]


**Sửa:** khởi tạo hàng đợi bằng `deque([kill])` (đưa chính `kill` vào), như ở cách 3.

### Lỗi 3 — Dùng `dict` thường và gặp `KeyError`

`g = {}` rồi `g[p].append(c)` lỗi ngay ở lần đầu một cha xuất hiện; còn khi duyệt, `g[u]` với `u` là **lá** (không có con) cũng lỗi.

In [31]:
def wrong_keyerror(pid, ppid, kill):
    g = {}
    for c, p in zip(pid, ppid):
        g[p].append(c)                   # SAI: g[p] chưa tồn tại
    return g

try:
    wrong_keyerror([1, 3, 10, 5], [3, 0, 5, 3], 5)
except KeyError as e:
    print("KeyError với khóa", e)

def fix_setdefault(pid, ppid):
    g = {}
    for c, p in zip(pid, ppid):
        g.setdefault(p, []).append(c)    # cách đúng 1: setdefault
    return g

g = fix_setdefault([1, 3, 10, 5], [3, 0, 5, 3])
print("sửa bằng setdefault:", g)
print("lá 10:", g.get(10, []))           # nhớ dùng .get khi đọc khóa có thể không tồn tại

KeyError với khóa 3
sửa bằng setdefault: {3: [1, 5], 0: [3], 5: [10]}
lá 10: []


**Cách đúng:** `defaultdict(list)` (tự tạo list rỗng) hoặc `setdefault`/`get(k, [])` như trên.

### Lỗi 4 — Nhầm **chỉ số** `i` với **id** `pid[i]`

`pid` và `ppid` là hai list **theo vị trí**, nhưng giá trị trong đó là **ID**, không phải chỉ số. Dùng `i` thay `pid[i]` khi dựng bảng cho ra kết quả sai (hoặc `IndexError`).

In [32]:
def wrong_index_vs_id(pid, ppid, kill):
    g = defaultdict(list)
    for i in range(len(pid)):
        g[ppid[i]].append(i)             # SAI: thêm chỉ số i, đúng phải là pid[i]
    ans, q = [], deque([kill])
    while q:
        u = q.popleft()
        ans.append(u)
        q.extend(g[u])
    return ans

pid, ppid, kill = [1, 3, 10, 5], [3, 0, 5, 3], 5
print("sai    :", wrong_index_vs_id(pid, ppid, kill), "| đúng:", bfs_queue(pid, ppid, kill))
assert sorted(wrong_index_vs_id(pid, ppid, kill)) != [5, 10]

sai    : [5, 2] | đúng: [5, 10]


**Vì sao sai?** Tiến trình `10` ở chỉ số `2`; thêm `2` vào hàng đợi rồi tra `g[2]` là sai nút. **Cách đúng:** `g[ppid[i]].append(pid[i])` hoặc `zip(pid, ppid)`.

### Lỗi 5 — Kết quả đặt ở mức class bị dính giữa các lần gọi

LeetCode gọi `killProcess` nhiều lần trên **cùng một đối tượng** (hoặc chạy nhiều test trong cùng tiến trình). Nếu `ans = []` nằm ở mức class thì phần tử của lần gọi trước còn nguyên ở lần gọi sau.

In [33]:
class WrongSolution:
    ans = []                              # SAI: biến mức class, dùng chung giữa các lần gọi

    def killProcess(self, pid, ppid, kill):
        g = defaultdict(list)
        for c, p in zip(pid, ppid):
            g[p].append(c)
        stk = [kill]
        while stk:
            u = stk.pop()
            self.ans.append(u)
            stk.extend(g[u])
        return self.ans

w = WrongSolution()
print("lần 1:", w.killProcess([1, 3, 10, 5], [3, 0, 5, 3], 5))
print("lần 2:", w.killProcess([1], [0], 1), "<- dính kết quả lần 1")
assert sorted(w.killProcess([1], [0], 1)) != [1]
print("đúng (cách 3) lần 2:", Solution().killProcess([1], [0], 1))

lần 1: [5, 10]
lần 2: [5, 10, 1] <- dính kết quả lần 1
đúng (cách 3) lần 2: [1]


**Cách đúng:** tạo `ans = []` **bên trong** hàm (biến cục bộ) mỗi lần gọi.

### Lỗi 6 — Đệ quy quá sâu trên cây dây chuyền

DFS đệ quy sâu bằng độ sâu của cây. Với dây chuyền ~3000 tầng, Python mặc định đã `RecursionError`; cây 5×10^4 tầng thì càng nặng. Hãy chọn BFS / DFS lặp.

In [34]:
m = 3000
cp, cpp = list(range(1, m + 1)), [0] + list(range(1, m))
try:
    dfs_recursive(cp, cpp, 1)
    print("DFS đệ quy chạy được (giới hạn đệ quy đang đủ lớn)")
except RecursionError:
    print("DFS đệ quy: RecursionError với dây chuyền", m, "tầng")

got = bfs_queue(cp, cpp, 1)
assert sorted(got) == cp
print("BFS: OK, bị kill", len(got), "tiến trình")

DFS đệ quy: RecursionError với dây chuyền 3000 tầng
BFS: OK, bị kill 3000 tiến trình


## 14. Mẹo & nhận dạng pattern

- **Dữ liệu cho dạng "(con, cha)" nhưng câu hỏi cần "con của cha"** → dựng **bảng đảo** `cha → [con]` bằng `defaultdict(list)`. Mẫu này xuất hiện ở rất nhiều bài (danh sách kề từ danh sách cạnh).
- **"Kill / xóa / lan truyền xuống cây con"** → duyệt từ nút gốc của cây con bằng BFS hoặc DFS; kết quả là cây con đó.
- **Thứ tự kết quả bất kỳ** → đừng tốn công khớp thứ tự; test bằng `sorted` hoặc hàm kiểm tra hợp lệ.
- **Cây không có chu trình** → không cần `visited`; nhưng nếu đề cho **đồ thị** tổng quát thì phải có.
- **Độ sâu có thể tới n** → ưu tiên BFS/stack; nếu cần đệ quy thì nâng `sys.setrecursionlimit` (nhưng có thể tràn stack hệ điều hành) — không an toàn bằng lặp.
- Tên biến: dùng `child`/`parent` (hoặc `c`/`p`) khi `zip(pid, ppid)` để tránh nhầm thứ tự `(con, cha)`.
- Kiểm tra nhanh sau khi viết: kill **gốc** (phải ra tất cả), kill **lá** (chỉ chính nó), `n = 1`.

## 15. Biến thể

### 15.1. Kill nhiều tiến trình cùng lúc

Cho `kills` là list ID. Duyệt BFS **đa nguồn**: đưa tất cả vào hàng đợi; vì hai tiến trình trong `kills` có thể là tổ tiên của nhau nên cần `visited` để không trả trùng.

In [35]:
def kill_many(pid, ppid, kills):
    g = defaultdict(list)
    for c, p in zip(pid, ppid):
        g[p].append(c)
    seen = set(kills)
    q = deque(seen)
    ans = []
    while q:
        u = q.popleft()
        ans.append(u)
        for v in g[u]:
            if v not in seen:
                seen.add(v)
                q.append(v)
    return ans

pid, ppid = [1, 2, 3, 4, 5, 6, 7], [0, 1, 1, 2, 2, 3, 3]
print(sorted(kill_many(pid, ppid, [2, 3])))
print(sorted(kill_many(pid, ppid, [1, 2])))      # 2 là con của 1: không trùng
assert sorted(kill_many(pid, ppid, [2, 3])) == [2, 3, 4, 5, 6, 7]
assert sorted(kill_many(pid, ppid, [1, 2])) == [1, 2, 3, 4, 5, 6, 7]

[2, 3, 4, 5, 6, 7]
[1, 2, 3, 4, 5, 6, 7]


### 15.2. Kết quả theo từng "đợt" (tầng)

Giả sử mỗi tiến trình mất 1 đơn vị thời gian để kill con của nó: đợt 0 là `kill`, đợt 1 là các con, đợt 2 là các cháu... BFS theo tầng cho ra list các list.

In [36]:
def kill_by_wave(pid, ppid, kill):
    g = defaultdict(list)
    for c, p in zip(pid, ppid):
        g[p].append(c)
    waves = []
    cur = [kill]
    while cur:
        waves.append(cur)
        nxt = []
        for u in cur:
            nxt.extend(g[u])
        cur = nxt
    return waves

print(kill_by_wave(pid, ppid, 1))
assert kill_by_wave(pid, ppid, 1) == [[1], [2, 3], [4, 5, 6, 7]]
assert kill_by_wave(pid, ppid, 7) == [[7]]

[[1], [2, 3], [4, 5, 6, 7]]


### 15.3. Chỉ đếm số tiến trình bị kill

Không cần trả danh sách: đếm là đủ (`len(...)`). Có thể tính **kích thước cây con** cho mọi nút bằng DP một lần nếu cần trả lời nhiều truy vấn: xử lý theo thứ tự ngược (con trước cha) và cộng dồn `size[cha] += size[con]` — mở rộng thú vị cho bạn tự thử.

### 15.4. Bài liên quan

- **LeetCode 690 — Employee Importance**: cũng "cha → con" rồi cộng giá trị cây con.
- **LeetCode 1376 — Time Needed to Inform All Employees**: tương tự nhưng tính **thời gian lan truyền** (độ sâu theo trọng số).
- **LeetCode 133, 200, 207**: duyệt đồ thị bằng BFS/DFS với `visited`.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét lại toàn bộ | mỗi nút bị kill quét cả `ppid` để tìm con | O(n·m) ≤ O(n²) | O(m) | chỉ để hiểu đề / `n` rất nhỏ |
| 2. Bảng + DFS đệ quy | dựng `g`, `dfs(u)` gọi con | O(n) | O(n) + độ sâu | cây nông, muốn code ngắn nhất |
| 3. Bảng + BFS `deque` | dựng `g`, duyệt từng tầng | O(n) | O(n) | **mặc định nên dùng**, an toàn |
| 4. Bảng + DFS ngăn xếp | như 2 nhưng tự quản stack | O(n) | O(n) | cần thứ tự DFS mà tránh đệ quy |
| 5. Tổ tiên có nhớ | hỏi từng nút "có `kill` làm tổ tiên không", memo | O(n) | O(n) | khi có nhiều truy vấn `kill` khác nhau trên cùng cây (dùng lại `parent`) |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng — **không được gọi** trước khi bạn viết xong. Đáp án ở mục 19.

### Bài 1 (đoán output)

Không chạy code, hãy đoán `result` rồi điền vào `my_guess_1` (một `dict` thường: khóa cha → list con, theo thứ tự thêm).

In [37]:
pid = [2, 4, 7, 9]
ppid = [0, 2, 2, 4]
g = defaultdict(list)
for child, parent in zip(pid, ppid):
    g[parent].append(child)
result = dict(g)
# print(result)   # <- bỏ dấu # sau khi đã đoán xong để kiểm tra

my_guess_1 = None    # TODO: điền dict bạn đoán, ví dụ {0: [...], ...}

### Bài 2 (tính tay)

Với `pid = [1,2,3,4,5,6]`, `ppid = [0,1,1,2,2,3]`, `kill = 2`: vẽ cây trên giấy, rồi viết **tập** các tiến trình bị kill (sắp tăng dần) vào `my_answer_2`.

In [38]:
my_answer_2 = None    # TODO: điền list đã sắp tăng dần

### Bài 3 (điền TODO)

Hoàn thành hàm BFS dưới đây: thay các `...` bằng biểu thức/câu lệnh đúng.

In [39]:
def exercise3(pid, ppid, kill):
    g = defaultdict(list)
    for child, parent in zip(pid, ppid):
        ...                          # TODO: thêm child vào danh sách con của parent
    ans = []
    q = deque(...)                   # TODO: hàng đợi khởi tạo bằng gì?
    while q:
        u = ...                      # TODO: lấy phần tử đầu hàng đợi
        ans.append(u)
        ...                          # TODO: đưa các con của u vào hàng đợi
    return ans

### Bài 4 (tự viết lại)

Tự viết hàm `exercise4(pid, ppid, kill)` dùng **DFS lặp bằng ngăn xếp** (không đệ quy), trả về list ID bị kill (thứ tự bất kỳ).

In [40]:
def exercise4(pid, ppid, kill):
    # TODO: viết lời giải
    pass

### Bài 5 (biến thể / mini challenge)

Viết `exercise5(pid, ppid, kill)` trả về **số lượng tiến trình bị kill** và **độ sâu lớn nhất** (số "đợt" lan truyền, tính `kill` là đợt 1) dưới dạng tuple `(count, waves)`. Ví dụ trên cây của bài 2 với `kill = 1` ra `(6, 3)`.

In [41]:
def exercise5(pid, ppid, kill):
    # TODO: viết lời giải, trả về (count, waves)
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: đi lần lượt từng cặp `(child, parent)` theo `zip`.
- Mức 2: `g[parent].append(child)` — mỗi cặp thêm `child` vào list của `parent`.
- Mức 3: các cặp là `(2,0)`, `(4,2)`, `(7,2)`, `(9,4)` → khóa `0`, `2`, `4`. Ghi từng list con theo thứ tự thêm.

**Bài 2**
- Mức 1: vẽ cây: gốc `1`, con của `1` là `2` và `3`.
- Mức 2: kill `2` thì chỉ con cháu của `2` bị kill.
- Mức 3: con của `2` là `4` và `5`; `4`, `5` không có con. Tập: `[2, 4, 5]`.

**Bài 3**
- Mức 1: nhìn cách 3 ở mục 7.2.
- Mức 2: bảng: `g[parent].append(child)`; hàng đợi khởi tạo bằng chính `kill`; lấy đầu hàng đợi bằng `popleft`.
- Mức 3: `g[parent].append(child)`; `q = deque([kill])`; `u = q.popleft()`; `q.extend(g[u])`.

**Bài 4**
- Mức 1: dựng bảng con như thường lệ.
- Mức 2: `stk = [kill]`; `while stk:` pop một nút, ghi, đẩy các con.
- Mức 3: `stk.extend(g[u])` là đủ vì thứ tự bất kỳ.

**Bài 5**
- Mức 1: duyệt theo từng tầng (list `cur` và `nxt` như mục 15.2).
- Mức 2: đếm số lần `cur` còn phần tử = số đợt; cộng `len(cur)` vào bộ đếm.
- Mức 3: `count += len(cur)`; `waves += 1` mỗi vòng; `cur = tất cả con của cur`.

## 19. Đáp án tham khảo

Chạy các cell sau để kiểm tra bằng `assert`.

In [42]:
# Bài 1
pid = [2, 4, 7, 9]
ppid = [0, 2, 2, 4]
g = defaultdict(list)
for child, parent in zip(pid, ppid):
    g[parent].append(child)
answer_1 = dict(g)
print("Bài 1:", answer_1)
assert answer_1 == {0: [2], 2: [4, 7], 4: [9]}

# Bài 2
answer_2 = oracle([1, 2, 3, 4, 5, 6], [0, 1, 1, 2, 2, 3], 2)
print("Bài 2:", answer_2)
assert answer_2 == [2, 4, 5]

Bài 1: {0: [2], 2: [4, 7], 4: [9]}
Bài 2: [2, 4, 5]


In [43]:
# Bài 3
def exercise3_solution(pid, ppid, kill):
    g = defaultdict(list)
    for child, parent in zip(pid, ppid):
        g[parent].append(child)
    ans = []
    q = deque([kill])
    while q:
        u = q.popleft()
        ans.append(u)
        q.extend(g[u])
    return ans

# Bài 4
def exercise4_solution(pid, ppid, kill):
    g = defaultdict(list)
    for child, parent in zip(pid, ppid):
        g[parent].append(child)
    ans = []
    stk = [kill]
    while stk:
        u = stk.pop()
        ans.append(u)
        stk.extend(g[u])
    return ans

# Bài 5
def exercise5_solution(pid, ppid, kill):
    g = defaultdict(list)
    for child, parent in zip(pid, ppid):
        g[parent].append(child)
    count, waves = 0, 0
    cur = [kill]
    while cur:
        count += len(cur)
        waves += 1
        nxt = []
        for u in cur:
            nxt.extend(g[u])
        cur = nxt
    return count, waves

for pid, ppid, kill, expected in examples:
    assert sorted(exercise3_solution(pid, ppid, kill)) == sorted(expected)
    assert sorted(exercise4_solution(pid, ppid, kill)) == sorted(expected)
assert exercise5_solution([1, 2, 3, 4, 5, 6], [0, 1, 1, 2, 2, 3], 1) == (6, 3)
assert exercise5_solution([1, 2, 3, 4, 5, 6], [0, 1, 1, 2, 2, 3], 2) == (3, 2)
assert exercise5_solution([1], [0], 1) == (1, 1)
print("Đáp án bài 3, 4, 5 đúng trên ví dụ")

rng = random.Random(19)
for _ in range(500):
    n = rng.randint(1, 20)
    pid, ppid = random_tree(n, 50, rng)
    kill = rng.choice(pid)
    want = oracle(pid, ppid, kill)
    assert sorted(exercise3_solution(pid, ppid, kill)) == want
    assert sorted(exercise4_solution(pid, ppid, kill)) == want
    count, waves = exercise5_solution(pid, ppid, kill)
    assert count == len(want) and 1 <= waves <= count
print("Đối chiếu ngẫu nhiên OK")

Đáp án bài 3, 4, 5 đúng trên ví dụ
Đối chiếu ngẫu nhiên OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (BFS, O(n) thời gian, O(n) bộ nhớ):**

```python
class Solution:
    def killProcess(self, pid: List[int], ppid: List[int], kill: int) -> List[int]:
        g = defaultdict(list)
        for child, parent in zip(pid, ppid):
            g[parent].append(child)          # đảo chiều: cha -> các con
        ans = []
        q = deque([kill])
        while q:
            u = q.popleft()
            ans.append(u)
            q.extend(g[u])                   # mỗi nút vào hàng đợi đúng một lần
        return ans
```

**Checklist trước khi submit:**

- [ ] Dựng bảng `cha → [con]` **một lần**, không quét lại `ppid` trong vòng duyệt.
- [ ] Bắt đầu từ chính `kill` (kết quả **gồm cả `kill`**).
- [ ] Dùng `defaultdict(list)` (hoặc `get`/`setdefault`) để không bị `KeyError` với lá.
- [ ] Dùng `pid[i]` (ID), không dùng chỉ số `i`; hoặc `zip(pid, ppid)` cho an toàn.
- [ ] `ans` là biến **cục bộ** của hàm, không đặt ở mức class.
- [ ] Cây có thể sâu 5×10^4: dùng BFS / stack, tránh đệ quy.
- [ ] Đã thử: `n = 1`, kill gốc, kill lá, cây dây chuyền, cây hình sao, `pid` không theo thứ tự cha–con.
- [ ] Test bằng `sorted(...)` vì thứ tự kết quả bất kỳ; độ phức tạp Time O(n), Space O(n).